# NOTE
- Using the extracted netlist (`extracted_netlist.spice`)

In [ ]:
def parse_spice(spice_text):
    # Step 1: Join continuation lines ('+') and strip empty/comment lines
    raw_lines = spice_text.splitlines()
    joined_lines = []
    for line in raw_lines:
        line = line.strip()
        if not line or line.startswith("*"):
            continue
        if line.startswith("+"):
            if joined_lines:
                joined_lines[-1] += " " + line[1:].strip()
        else:
            joined_lines.append(line)

    # Step 2: Build the hierarchy tree
    tree = {"subckts": {}, "top_instances": []}
    current_subckt = None

    for line in joined_lines:
        tokens = line.split()
        cmd = tokens[0].lower()

        # Begin subcircuit definition
        if cmd == ".subckt":
            subckt_name = tokens[1]
            pins = tokens[2:]
            current_subckt = subckt_name
            tree["subckts"][subckt_name] = {"pins": pins, "instances": []}

        # End subcircuit definition
        elif cmd == ".ends":
            current_subckt = None

        # Component/Cell Instantiation (lines starting with X or M)
        elif tokens[0].upper().startswith(("X", "M")):
            inst_name = tokens[0]

            nets_and_cell = []
            params = {}

            # Separate pin connections/cell type from 'key=value' parameters
            for token in tokens[1:]:
                if "=" in token:
                    k, v = token.split("=", 1)
                    params[k] = v
                else:
                    nets_and_cell.append(token)

            # Last non-parameter token is the subckt/primitive type
            cell_type = nets_and_cell[-1] if nets_and_cell else None
            connections = nets_and_cell[:-1]

            inst_dict = {
                "name": inst_name,
                "type": cell_type,
                "connections": connections,
                "params": params,
            }

            if current_subckt:
                tree["subckts"][current_subckt]["instances"].append(inst_dict)
            else:
                tree["top_instances"].append(inst_dict)

    return tree

In [ ]:
'''
We just want the top level netlist of the 'puzzle' block
.subckt sky130_fd_sc_hd__or4bb_2 A X B D_N C_N VPWR VGND VPB VNB
'''
with open(r'/content/extracted_netlist__2026_08_10.spice') as f:
    text = f.read()
    idx=text.find('.subckt puzzle')
    top_netlist = text[idx:]
# top_netlist
# Just a dict for quick cell lookup based on name
tree = parse_spice(top_netlist)
# print(tree)
cell_lookup = {cell_dict['name']:cell_dict for cell_dict in tree['subckts']['puzzle']['instances']}


Pick out just one instance

In [ ]:
tree['subckts']['puzzle']['instances'][0]

In [ ]:
cell_lookup = {cell_dict['name']:cell_dict for cell_dict in tree['subckts']['puzzle']['instances']}
#cell_lookup['Xsky130_fd_sc_hd__a211oi_2_1']

# .index('Xsky130_fd_sc_hd__nor4_2_0')

In [ ]:
cell_lookup[list(cell_lookup.keys())[0]]

In [ ]:
#cell_lookup['Xsky130_fd_sc_hd__nor4_2_0']

In [ ]:
len(cell_lookup)

- Flat netlist: `puzzle` is flat (has no additional level of hierarchy beyind just the standard cells) with 942 cells (no subckts)
- There are 68 standard cell subckts used in building the `puzzle` top cell
- Leading X in cellname declarations: Declarations start with 'X' and all other mentions of the same cell in the netlist dont hav this leading 'X'. eg.
The declaration for this 9 fanout MUX is:
```
Xsky130_fd_sc_hd__mux2_1_9 sky130_fd_sc_hd__inv_2_7/A sky130_fd_sc_hd__mux2_1_9/A1
```
And all other mentions of this particular MUX go without the X
```
Xsky130_fd_sc_hd__mux2_1_16 sky130_fd_sc_hd__inv_2_7/A sky130_fd_sc_hd__mux2_1_9/A0 <---- here is our MUX!
```
(Maybe this leading X is a ngspice convention?...)

In [1]:
from netlist_to_verilog import *
in_path, out_path = 'inputs/extracted_netlist.spice', 'inputs/puzzle.v'
forced_top = None

lib_ports, subckt_bodies, instantiated_types = parse_spice(in_path)
top_name = find_top(lib_ports, instantiated_types, forced_top)
top_ports = lib_ports[top_name]
top_ports = [p for p in top_ports if p not in POWER_PINS]

instances = parse_instances(subckt_bodies[top_name])

instances, clk_alias_map, removed = collapse_clock_trees(lib_ports, instances)
if removed:
    print(f"Collapsed {len(removed)} clock-buffer cells into "
          f"{len(set(clk_alias_map.values()))} canonical clock net(s): "
          f"{sorted(set(clk_alias_map.values()))}")

assigns, flops, ties = build_model(lib_ports, instances)
sanity_check(assigns, flops, ties)

verilog = generate_verilog(top_name, top_ports, assigns, flops, ties)
with open(out_path, "w") as f:
    f.write(verilog)

print(f"Top module   : {top_name}")
print(f"Instances    : {len(instances)}")
print(f"Comb. gates  : {len(assigns)}")
print(f"Flip-flops   : {len(flops)}")
print(f"Tie cells    : {len(ties)}")
print(f"Written to   : {out_path}")



Collapsed 32 clock-buffer cells into 1 canonical clock net(s): ['clk']
Top module   : puzzle
Instances    : 910
Comb. gates  : 598
Flip-flops   : 92
Tie cells    : 12
Written to   : inputs/puzzle.v
